# MobileADAS3D-H1 v2 tiny-subset overfit gate

This is a controlled memorization test on 16 audited Chen-training images. It is not a benchmark. It must prove matched confidence, background suppression, localization, and object-count control before another full KITTI run. Distillation is disabled.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
from collections import deque
import hashlib, json, os, shlex, subprocess, sys
REPO_URL='https://github.com/Ali-RT/mobile_adas3d.git'; BRANCH='main'
PROJECT_DIR=Path('/content/mobile_adas3d')
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti'); DATASET_VIEW=Path('/content/kitti_h1')
FULL_SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
TINY_SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/h1_v2_tiny16')
OUTPUT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_outputs/mobileadas3d_h1_v2_tiny16')
CONFIG=PROJECT_DIR/'configs/kitti_mobileadas3d_h1_v2_tiny_overfit.yaml'
RUN_NAME='mobileadas3d_h1_v2_tiny16'
def run(command,cwd=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    result=subprocess.run(command,cwd=cwd)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_streamed(command,cwd,log_path,allow_failure=False):
    command=[str(x) for x in command]; log_path.parent.mkdir(parents=True,exist_ok=True)
    print('+',shlex.join(command),flush=True); print('Durable log:',log_path,flush=True)
    tail=deque(maxlen=160); env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=p.wait()
    if code and not allow_failure: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return code


In [ ]:
# Refresh the repository and require a GPU. Make the local implementation commit available remotely first.
if not (PROJECT_DIR/'.git').exists(): run(['git','clone','--branch',BRANCH,REPO_URL,PROJECT_DIR])
else:
    run(['git','fetch','origin'],PROJECT_DIR); run(['git','checkout',BRANCH],PROJECT_DIR); run(['git','pull','--ff-only','origin',BRANCH],PROJECT_DIR)
os.chdir(PROJECT_DIR)
run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],PROJECT_DIR)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Choose Runtime > Change runtime type > GPU')
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_DIR,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# Resolve KITTI and create a deterministic 16-image split containing both product classes.
def count_files(path,suffix): return sum(1 for p in path.iterdir() if p.is_file() and p.suffix==suffix) if path.is_dir() else 0
def complete(root): return count_files(root/'training/image_2','.png')==7481 and count_files(root/'training/label_2','.txt')==7481 and count_files(root/'training/calib','.txt')==7481
if complete(LOCAL_DATASET_ROOT): DATASET_ROOT=LOCAL_DATASET_ROOT
else:
    aliases={'image_2':['image_2','image_02'],'label_2':['label_2','label_02'],'calib':['calib']}
    (DATASET_VIEW/'training').mkdir(parents=True,exist_ok=True)
    for canonical,candidates in aliases.items():
        source=next((DRIVE_DATASET_ROOT/'training'/name for name in candidates if (DRIVE_DATASET_ROOT/'training'/name).is_dir()),None)
        if source is None: raise FileNotFoundError(f'Missing source for {canonical}')
        link=DATASET_VIEW/'training'/canonical
        if not link.exists() and not link.is_symlink(): link.symlink_to(source,target_is_directory=True)
    DATASET_ROOT=DATASET_VIEW
if not complete(DATASET_ROOT): raise RuntimeError(f'KITTI view incomplete: {DATASET_ROOT}')
full_ids=[x.strip() for x in (FULL_SPLIT_DIR/'train.txt').read_text().splitlines() if x.strip()]
ped=[]; other=[]
for sample_id in full_ids:
    names={line.split()[0] for line in (DATASET_ROOT/'training/label_2'/f'{sample_id}.txt').read_text().splitlines() if line.strip()}
    (ped if names & {'Pedestrian','Person_sitting'} else other).append(sample_id)
tiny_ids=ped[:8]+other[:8]
if len(tiny_ids)!=16: raise RuntimeError(f'Expected 16 tiny samples, got {len(tiny_ids)}')
TINY_SPLIT_DIR.mkdir(parents=True,exist_ok=True)
payload='\n'.join(tiny_ids)+'\n'
for name in ('train.txt','val.txt'): (TINY_SPLIT_DIR/name).write_text(payload)
manifest={'schema_version':1,'purpose':'H1-v2 memorization only','samples':tiny_ids,'split_sha256':hashlib.sha256(payload.encode()).hexdigest(),'distillation_enabled':False}
(TINY_SPLIT_DIR/'tiny16_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print(json.dumps(manifest,indent=2)); print('Dataset root:',DATASET_ROOT)


In [ ]:
# Validate the objective and run a real CUDA forward/loss preflight.
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
COMMON=['--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',TINY_SPLIT_DIR,'--output-dir',OUTPUT_DIR]
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_h1_set_training.py','-v'],PROJECT_DIR)
run_streamed([sys.executable,'-u','scripts/check_training_ready.py','--config',CONFIG,*COMMON,'--require-cuda','--report',OUTPUT_DIR/'training_preflight.json'],PROJECT_DIR,OUTPUT_DIR/'training_preflight.log')
from tools.config import load_config
cfg=load_config(str(CONFIG))
assert cfg['loss']['classification_mode']=='implicit_background_softmax'
assert cfg['distillation']['enabled'] is False and cfg['training']['epochs']==100
print('H1-v2 preflight passed; distillation=false')


In [ ]:
# Train or resume the exact tiny run. This is only 400 optimizer steps at batch size 4.
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
RESUME=candidates[-1] if candidates else None
command=[sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',CONFIG,*COMMON,'--run-name',RUN_NAME]
completed=False
if RESUME:
    state=torch.load(RESUME,map_location='cpu',weights_only=False); completed=state.get('epoch',0)>=100
    if not completed: command += ['--resume',RESUME]
    else: print('Tiny run already complete:',RESUME)
if not completed: run_streamed(command,PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'train_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
if not candidates: raise FileNotFoundError('No tiny-run checkpoint')
LATEST_CHECKPOINT=candidates[-1]; TRAIN_RUN_DIR=LATEST_CHECKPOINT.parent.parent
state=torch.load(LATEST_CHECKPOINT,map_location='cpu',weights_only=False)
if state.get('epoch')!=100: raise RuntimeError(f'Expected epoch 100, got {state.get("epoch")}')
print('Run:',TRAIN_RUN_DIR); print('Checkpoint:',LATEST_CHECKPOINT)


In [ ]:
# Hard query-separation gate. Failure is expected to stop here for review.
REPORT=TRAIN_RUN_DIR/'h1_v2_tiny_query_diagnostics.json'
code=run_streamed([sys.executable,'-u','scripts/diagnose_h1_queries.py','--config',CONFIG,*COMMON,'--checkpoint',LATEST_CHECKPOINT,'--split','val','--score-threshold','0.1','--report',REPORT],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/'tiny_query_diagnostics.log',allow_failure=True)
report=json.loads(REPORT.read_text())
print(json.dumps(report,indent=2))
if code: print('STOP: tiny-overfit gate failed. Send this report; do not start a full run.')
else: print('PASS: send this report before preparing another full KITTI run.')
